In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels

In [12]:
CDS_data = pd.read_csv('data/final_sample.csv')
CDS_data['Date'] = pd.to_datetime(CDS_data['Date'])
CDS_data.set_index('Date', inplace=True)

Global_Controls = pd.read_excel('data/raw/data_masterfile.xlsx', sheet_name='Global_Controls')
Global_Controls = Global_Controls.rename(columns={'Dates': 'Date'})
Global_Controls['Date'] = pd.to_datetime(Global_Controls['Date'])
Global_Controls.set_index('Date', inplace=True)

merged = pd.merge(CDS_data, Global_Controls, on='Date', how='inner')
merged = merged.reset_index()
merged.head()

,Date,Brent,OVX,Saudi Arabia,Qatar,Kazakhstan,Bahrain,Colombia,Mexico,Turkey,...,Poland,SPX Index,MXEF Index,VIX Index,MOVE Index,DXY Curncy,JPEIDIVR Index,USGG10YR Index,USGG2YR Index,USGG30YR Index
0,2018-01-05,67.620003,22.299999,90.16,99.77,110.14,288.67,96.54999,97.04,157.63,...,55.64,2743.15,1201.01,9.22,46.5962,91.949,862.96,2.4763,1.9599,2.8105
1,2018-01-12,69.870003,23.520000,84.75,92.39,104.90,281.66,92.12000,97.29,155.20,...,52.31,2786.24,1208.17,10.16,46.0677,90.974,859.84,2.5462,1.9976,2.8491
2,2018-01-19,68.610001,21.200001,85.25,94.94,105.40,241.87,91.14000,99.99,167.68,...,50.26,2810.30,1232.60,11.27,47.6086,90.572,858.13,2.6592,2.0647,2.9331
3,2018-01-26,70.519997,22.290001,85.24,89.69,95.09,244.64,88.17000,98.02,163.09,...,49.55,2872.87,1273.07,11.08,51.9902,89.067,860.56,2.6599,2.1163,2.9114
4,2018-02-02,68.580002,23.799999,78.35,86.53,93.51,240.78,92.84999,99.01,165.47,...,49.84,2762.13,1230.84,17.31,55.6349,89.195,853.32,2.8411,2.1413,3.0868


## Setting up the regression

### Reshaping data to panel data format

In [44]:
oil_exporters = ['Saudi Arabia', 'Qatar', 'Kazakhstan', 'Bahrain', 'Colombia', 'Mexico', 'Brazil', 'Egypt']
controls = ['Turkey', 'South Africa', 'Chile', 'Philippines', 'Indonesia', 'Poland']
# Long format
panel_data = []

for country in oil_exporters + controls:
    temp = merged[['Date', 'Brent', 'OVX', 'VIX Index', country]].copy()
    temp['Country'] = country
    temp['CDS'] = temp[country]
    temp['OilExporter'] = 1 if country in oil_exporters else 0
    temp = temp[['Date', 'Country', 'CDS', 'Brent', 'OVX', 'VIX Index', 'OilExporter']]
    panel_data.append(temp)

panel = pd.concat(panel_data, ignore_index=True)
panel = panel.sort_values(by=['Country', 'Date']).reset_index(drop=True)

panel['CDS_ret'] = panel.groupby('Country')['CDS'].pct_change()
panel['Oil_ret'] = panel['Brent'].pct_change()
panel['OVX_chg'] = panel['OVX'].pct_change()
panel['VIX_chg'] = panel['VIX Index'].pct_change()
panel['OVX x OilExporter'] = panel['OVX_chg'] * panel['OilExporter']


panel = panel.dropna()
panel.head()

,Date,Country,CDS,Brent,OVX,VIX Index,OilExporter,CDS_ret,Oil_ret,OVX_chg,VIX_chg,OVX x OilExporter
1,2018-01-12,Bahrain,281.66,69.870003,23.520000,10.16,1,-0.024284,0.033274,0.054709,0.101952,0.054709
2,2018-01-19,Bahrain,241.87,68.610001,21.200001,11.27,1,-0.141270,-0.018034,-0.098639,0.109252,-0.098639
3,2018-01-26,Bahrain,244.64,70.519997,22.290001,11.08,1,0.011452,0.027838,0.051415,-0.016859,0.051415
4,2018-02-02,Bahrain,240.78,68.580002,23.799999,17.31,1,-0.015778,-0.027510,0.067743,0.562274,0.067743
5,2018-02-09,Bahrain,247.22,62.790001,29.850000,29.06,1,0.026746,-0.084427,0.254202,0.678798,0.254202


## Panel regressions

In [45]:
print("\n" + "="*70)
print("MODEL 1: Oil  +  Controls only")
print("="*70)

X1 = sm.add_constant(panel[['VIX_chg','OVX_chg']])
model1 = sm.OLS(panel['CDS_ret'], X1).fit(cov_type='cluster', cov_kwds={'groups': panel['Country']})
print(model1.summary())



MODEL 1: Oil  +  Controls only
                            OLS Regression Results                            
Dep. Variable:                CDS_ret   R-squared:                       0.199
Model:                            OLS   Adj. R-squared:                  0.199
Method:                 Least Squares   F-statistic:                     78.86
Date:                Thu, 29 Jan 2026   Prob (F-statistic):           5.38e-08
Time:                        13:52:36   Log-Likelihood:                 5946.1
No. Observations:                5096   AIC:                        -1.189e+04
Df Residuals:                    5093   BIC:                        -1.187e+04
Df Model:                           2                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.000

In [46]:
print("\n" + "="*70)
print("MODEL 2: Oil  +  Dummy")
print("="*70)

X1 = sm.add_constant(panel[['OVX_chg','OVX x OilExporter', 'VIX_chg']])
model1 = sm.OLS(panel['CDS_ret'], X1).fit(cov_type='cluster', cov_kwds={'groups': panel['Country']})
print(model1.summary())



MODEL 2: Oil  +  Dummy
                            OLS Regression Results                            
Dep. Variable:                CDS_ret   R-squared:                       0.200
Model:                            OLS   Adj. R-squared:                  0.200
Method:                 Least Squares   F-statistic:                     89.41
Date:                Thu, 29 Jan 2026   Prob (F-statistic):           6.25e-09
Time:                        13:52:36   Log-Likelihood:                 5948.4
No. Observations:                5096   AIC:                        -1.189e+04
Df Residuals:                    5092   BIC:                        -1.186e+04
Df Model:                           3                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const         

## Quantile regressions

In [55]:
print("="*70)
print("DESIGN 1: EXTREME OIL MOVES ONLY (top/bottom 10%)")
print("="*70)

# Define extreme oil moves

for pct in [0.10, 0.05, 0.01, 0.005]:


    oil_qtl = panel['OVX_chg'].quantile(1-pct)

    extreme_panel = panel[(panel['OVX_chg'] > oil_qtl)]

    print(f"Extreme observations: {len(extreme_panel)} ({len(extreme_panel)/len(panel)*100:.1f}%)")

    X = sm.add_constant(extreme_panel[['OVX_chg', 'VIX_chg', 'OVX x OilExporter']])
    model = sm.OLS(extreme_panel['CDS_ret'], X).fit(cov_type='cluster', cov_kwds={'groups': extreme_panel['Country']})
    print(model.summary())

DESIGN 1: EXTREME OIL MOVES ONLY (top/bottom 10%)
Extreme observations: 504 (9.9%)
                            OLS Regression Results                            
Dep. Variable:                CDS_ret   R-squared:                       0.255
Model:                            OLS   Adj. R-squared:                  0.251
Method:                 Least Squares   F-statistic:                     53.34
Date:                Thu, 29 Jan 2026   Prob (F-statistic):           1.45e-07
Time:                        14:17:11   Log-Likelihood:                 275.29
No. Observations:                 504   AIC:                            -542.6
Df Residuals:                     500   BIC:                            -525.7
Df Model:                           3                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------